# Alethia Feature Validation

Tests whether the demographic SAE features in the Alethia repo's `constants.py` actually encode sex/age.

**Method**: Compare four conditions per patient case:
- **[A]** Plain text prompt WITH demographics (ground truth)
- **[B]** No demographics + set-value clamping (sets features to demographic mean)
- **[C]** No demographics + multiplicative 5x clamping (the repo's current method)
- **[D]** No demographics, no clamping (neutral baseline)

If features are correct, [B] should produce similar diagnoses to [A].

**Key bugs found in the repo:**
1. Hook name mismatch: `scorer.py` hooks at `blocks.17.hook_out.hook_sae_acts_post` but SAE is layer 20
2. Multiplicative clamping on near-zero activations is a no-op
3. Feature indices in `constants.py` don't match the feature finder's output

**Layer selection rationale** (see docstring in `validate_features.py` for full citations):
- Arad et al. (2025, arXiv:2505.20063, EMNLP): Output features for steering emerge after ~50% depth
- Lieberum et al. (2024, arXiv:2408.05147): GemmaScope canonical SAEs for 9B at layers 9, 20, 31, 41
- Layer 20 = 47.6% depth, within the recommended range

## 0. Setup

In [1]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

In [2]:
!pip install -q transformer_lens sae_lens jinja2 tqdm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.7/963.7 kB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.6/296.6 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.6/236.6 kB 28.4 MB/s eta 0:00:00


In [3]:
import os

# Set your Hugging Face token (required for Gemma)
os.environ["HF_TOKEN"] = ""  # <-- paste your token here

# If running from the repo root, data/ folder should be accessible
# If you cloned the repo, the paths below should work as-is
DATA_DIR = "data"

In [12]:
!git clone https://github.com/Amelia3141/alethia-feature-testing.git
%cd alethia-feature-testing

Cloning into 'alethia-feature-testing'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 393.08 KiB | 6.55 MiB/s, done.
/content/alethia-feature-testing


## 1. Configuration

In [4]:
import json
import ast
import time
import torch
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional

MODEL_NAME = "google/gemma-2-9b-it"
SAE_RELEASE = "gemma-scope-9b-it-res-canonical"
SAE_ID = "layer_20/width_16k/canonical"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Start small -- each case runs ~200 forward passes
NUM_TEST_CASES = 5

DATA_PATH = os.path.join(DATA_DIR, "release_test_patients_mini_version.txt")
CONDITIONS_PATH = os.path.join(DATA_DIR, "release_conditions.json")
EVIDENCES_PATH = os.path.join(DATA_DIR, "release_evidences.json")

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"SAE: {SAE_RELEASE} / {SAE_ID}")

Device: cuda
Model: google/gemma-2-9b-it
SAE: gemma-scope-9b-it-res-canonical / layer_20/width_16k/canonical


## 2. Features from the repo's constants.py

These are the features hardcoded in the Alethia repo. Each maps `feature_idx -> {demographic mean activation values}`.

In [5]:
MALE_FEATURES_WITH_DIRECTIONS = {
    12593: {"male_mean": -0.346, "female_mean": 0.256},
    11208: {"male_mean": 0.321, "female_mean": -0.224},
    13522: {"male_mean": 0.319, "female_mean": -0.388},
    1832: {"male_mean": 0.306, "female_mean": 0.0},
    8718: {"male_mean": 0.293, "female_mean": 0.0},
    1476: {"male_mean": -0.0912, "female_mean": -0.0972},
    1997: {"male_mean": 0.0369, "female_mean": 0.0153},
    793: {"male_mean": -0.0787, "female_mean": -0.0796},
    728: {"male_mean": 0.2034, "female_mean": 0.1959},
    126: {"male_mean": -0.1319, "female_mean": -0.1347},
    238: {"male_mean": -0.1698, "female_mean": -0.1822},
    1202: {"male_mean": -0.3500, "female_mean": -0.3666},
    1738: {"male_mean": -0.1565, "female_mean": -0.1678},
    317: {"male_mean": -0.0045, "female_mean": -0.0106},
    356: {"male_mean": -0.4624, "female_mean": -0.4880},
}

FEMALE_FEATURES_WITH_DIRECTIONS = {
    13522: {"male_mean": -0.319, "female_mean": 0.388},
    1975: {"male_mean": 0.0, "female_mean": 0.309},
    12593: {"male_mean": 0.346, "female_mean": -0.256},
    10863: {"male_mean": 0.299, "female_mean": -0.243},
    11208: {"male_mean": -0.321, "female_mean": 0.224},
    953: {"male_mean": -0.1137, "female_mean": -0.1029},
    694: {"male_mean": -0.3857, "female_mean": -0.3635},
    696: {"male_mean": 0.1301, "female_mean": 0.1490},
    346: {"male_mean": -0.2348, "female_mean": -0.2233},
    861: {"male_mean": 0.1616, "female_mean": 0.1646},
    1989: {"male_mean": -0.1159, "female_mean": -0.1047},
    610: {"male_mean": 0.1145, "female_mean": 0.1256},
    486: {"male_mean": 0.0266, "female_mean": 0.0440},
    1899: {"male_mean": 0.1243, "female_mean": 0.1445},
    311: {"male_mean": -0.0378, "female_mean": -0.0330},
}

YOUNG_FEATURES_WITH_DIRECTIONS = {
    11208: {"young_mean": 0.537, "old_mean": -0.468},
    5547: {"young_mean": -0.535, "old_mean": 0.496},
    158: {"young_mean": 0.509, "old_mean": -0.439},
    778: {"young_mean": 0.365, "old_mean": -0.350},
    10863: {"young_mean": -0.299, "old_mean": 0.446},
}

OLD_FEATURES_WITH_DIRECTIONS = {
    5547: {"old_mean": -0.496, "young_mean": 0.535},
    11208: {"old_mean": 0.468, "young_mean": -0.537},
    10863: {"old_mean": -0.446, "young_mean": 0.299},
    10327: {"old_mean": -0.309, "young_mean": 0.0},
    11587: {"old_mean": 0.288, "young_mean": 0.0},
}

print(f"Male features: {len(MALE_FEATURES_WITH_DIRECTIONS)}")
print(f"Female features: {len(FEMALE_FEATURES_WITH_DIRECTIONS)}")
print(f"Young features: {len(YOUNG_FEATURES_WITH_DIRECTIONS)}")
print(f"Old features: {len(OLD_FEATURES_WITH_DIRECTIONS)}")

# Check overlap
sex_overlap = set(MALE_FEATURES_WITH_DIRECTIONS) & set(FEMALE_FEATURES_WITH_DIRECTIONS)
age_overlap = set(YOUNG_FEATURES_WITH_DIRECTIONS) & set(OLD_FEATURES_WITH_DIRECTIONS)
cross_overlap = (set(MALE_FEATURES_WITH_DIRECTIONS) | set(FEMALE_FEATURES_WITH_DIRECTIONS)) & \
                (set(YOUNG_FEATURES_WITH_DIRECTIONS) | set(OLD_FEATURES_WITH_DIRECTIONS))
print(f"\nSex feature overlap (male & female): {sex_overlap}")
print(f"Age feature overlap (young & old): {age_overlap}")
print(f"Cross-demographic overlap (sex & age): {cross_overlap}")
if cross_overlap:
    print("WARNING: Features shared across demographics will conflate effects")

Male features: 15
Female features: 15
Young features: 5
Old features: 5

Sex feature overlap (male & female): {11208, 12593, 13522}
Age feature overlap (young & old): {11208, 5547, 10863}
Cross-demographic overlap (sex & age): {11208, 10863}


## 3. Load model, SAE, and data

In [7]:
from sae_lens import SAE
from transformer_lens import HookedTransformer

print(f"Loading model: {MODEL_NAME} on {DEVICE}...")
model = HookedTransformer.from_pretrained_no_processing(
    model_name=MODEL_NAME,
    torch_dtype=DTYPE,
    device=DEVICE,
    move_to_device=True,
)
model.eval()
print(f"Model loaded. {model.cfg.n_layers} layers.")

print(f"\nLoading SAE: {SAE_RELEASE} / {SAE_ID}...")
sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=DEVICE,
)
sae.eval()
HOOK_NAME = getattr(sae.cfg, 'hook_name', None) or getattr(sae.cfg, 'hook_point', None) or 'blocks.20.hook_resid_post'
print(f"SAE loaded. Hook: {HOOK_NAME}, Features: {sae.W_enc.shape[1]}")
print(f"\nNOTE: The repo's scorer.py uses 'blocks.17.hook_out.hook_sae_acts_post'")
print(f"      which is WRONG for this layer-20 SAE. We use: {HOOK_NAME}")

Loading model: google/gemma-2-9b-it on cuda...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-9b-it into HookedTransformer
Model loaded. 42 layers.

Loading SAE: gemma-scope-9b-it-res-canonical / layer_20/width_16k/canonical...
SAE loaded. Hook: blocks.20.hook_resid_post, Features: 16384

NOTE: The repo's scorer.py uses 'blocks.17.hook_out.hook_sae_acts_post'
      which is WRONG for this layer-20 SAE. We use: blocks.20.hook_resid_post


In [16]:
%cd alethia-feature-validation
# Load patient data
df = pd.read_csv(DATA_PATH)
cases = []
for _, row in df.iterrows():
    cases.append({
        "age": row.get("age", row.get("AGE")),
        "sex": row.get("sex", row.get("SEX")),
        "diagnosis": row.get("diagnosis", row.get("PATHOLOGY")),
        "features": row.get("features", row.get("EVIDENCES")),
    })

with open(CONDITIONS_PATH, "r") as f:
    conditions_data = json.load(f)
diagnosis_list = [v["condition_name"] for v in conditions_data.values()]

with open(EVIDENCES_PATH, "r") as f:
    evidences = json.load(f)

print(f"Loaded {len(cases)} cases, {len(diagnosis_list)} candidate diagnoses")
print(f"Sample diagnoses: {diagnosis_list[:5]}")

/content/alethia-feature-testing/alethia-feature-validation
Loaded 2611 cases, 49 candidate diagnoses
Sample diagnoses: ['Spontaneous pneumothorax', 'Cluster headache', 'Boerhaave', 'Spontaneous rib fracture', 'GERD']


## 4. Helper functions

In [21]:
def symptoms_to_text(symptom_codes_str, evidences):
    """Convert symptom codes to natural language text."""
    if isinstance(symptom_codes_str, str):
        try:
            symptom_codes = ast.literal_eval(symptom_codes_str)
        except (ValueError, SyntaxError):
            return symptom_codes_str
    elif isinstance(symptom_codes_str, list):
        symptom_codes = symptom_codes_str
    else:
        return str(symptom_codes_str)

    parts = []
    for c in symptom_codes:
        if "@" not in c:
            name, value = c, ""
        else:
            name, value = c.split("_@_")

        if name not in evidences:
            parts.append(f"(Unknown: {c})")
            continue

        ev = evidences[name]
        if ev["data_type"] == "B":
            nl_value = "Yes."
        elif ev["data_type"] == "M":
            nl_value = ev["value_meaning"].get(value, {}).get("en", value)
        elif ev["data_type"] == "C":
            nl_value = f"{value} out of {ev['possible-values'][-1]}."
        else:
            nl_value = value

        question = "Q: " + ev["question_en"]
        answer = "A: " + nl_value
        parts.append(f"({question} {answer})")

    return str(parts).replace("'", "")


def score_all_diagnoses(
    prompt, model, diagnosis_list, sae=None,
    clamp_features=None, clamp_method="none", clamping_level=1.0,
):
    """Score all candidate diagnoses for a prompt.

    clamp_method: 'set_value', 'multiplicative', or 'none'
    clamp_features: dict of {feature_idx: target_value}
    """
    scores = {}

    for dx in diagnosis_list:
        full_text = f"{prompt} {dx}"
        toks_prefix = model.to_tokens(prompt)
        toks_full = model.to_tokens(full_text)

        min_len = min(toks_prefix.shape[-1], toks_full.shape[-1])
        shared = toks_prefix[0, :min_len] == toks_full[0, :min_len]
        prefix_len = int(min_len) if shared.all() else int(shared.int().argmin().item())

        eos_id = model.tokenizer.eos_token_id
        if eos_id is not None and toks_full[0, -1].item() == eos_id:
            dx_token_ids = toks_full[0, prefix_len:-1]
        else:
            dx_token_ids = toks_full[0, prefix_len:]

        if len(dx_token_ids) == 0:
            scores[dx] = float("-inf")
            continue

        with torch.no_grad():
            if clamp_method != "none" and clamp_features and sae is not None:
                hook_name = HOOK_NAME

                if clamp_method == "set_value":
                    def set_value_hook(activations, hook, features=clamp_features):
                        for pos in range(activations.shape[1]):
                            acts_at_pos = activations[:, pos, :]
                            sae_acts = sae.encode(acts_at_pos)
                            for feat_idx, target_val in features.items():
                                sae_acts[:, feat_idx] = target_val
                            reconstructed = sae.decode(sae_acts)
                            activations[:, pos, :] = reconstructed
                        return activations
                    logits = model.run_with_hooks(
                        toks_full, fwd_hooks=[(hook_name, set_value_hook)]
                    )

                elif clamp_method == "multiplicative":
                    def mult_hook(activations, hook, features=clamp_features, level=clamping_level):
                        for pos in range(activations.shape[1]):
                            acts_at_pos = activations[:, pos, :]
                            sae_acts = sae.encode(acts_at_pos)
                            for feat_idx in features:
                                sae_acts[:, feat_idx] *= level
                            reconstructed = sae.decode(sae_acts)
                            activations[:, pos, :] = reconstructed
                        return activations
                    logits = model.run_with_hooks(
                        toks_full, fwd_hooks=[(hook_name, mult_hook)]
                    )
            else:
                logits = model(toks_full)

        log_probs = []
        for i, tok_id in enumerate(dx_token_ids):
            pos = prefix_len - 1 + i
            logits_at_pos = logits[0, pos, :]
            lp = torch.nn.functional.log_softmax(logits_at_pos, dim=-1)[int(tok_id)]
            log_probs.append(lp.item())

        scores[dx] = sum(log_probs) / len(log_probs)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def jaccard(a, b):
    sa, sb = set(a), set(b)
    if len(sa | sb) == 0:
        return 0.0
    return len(sa & sb) / len(sa | sb)

## 5. Feature activation diagnostic

Before running the full validation, check whether the claimed features are even active. If they're all near zero, multiplicative clamping will do nothing regardless.

In [22]:
all_sex_features = set(MALE_FEATURES_WITH_DIRECTIONS) | set(FEMALE_FEATURES_WITH_DIRECTIONS)
all_age_features = set(YOUNG_FEATURES_WITH_DIRECTIONS) | set(OLD_FEATURES_WITH_DIRECTIONS)
all_features = all_sex_features | all_age_features
hook_name = HOOK_NAME

test_cases = [c for c in cases if c.get("age") and c.get("sex")][:3]

for i, case in enumerate(test_cases):
    age = case["age"]
    sex_word = "male" if case["sex"] == "M" else "female"
    symptoms_text = symptoms_to_text(case["features"], evidences)

    prompt_with = f"A {sex_word} {int(age)}-year-old patient has symptoms: {symptoms_text}."
    prompt_without = f"A patient has symptoms: {symptoms_text}."

    with torch.no_grad():
        toks = model.to_tokens(prompt_with)
        _, cache = model.run_with_cache(toks, return_type=None)
        sae_acts_with = sae.encode(cache[hook_name][0, -1, :].unsqueeze(0))[0]

        toks = model.to_tokens(prompt_without)
        _, cache = model.run_with_cache(toks, return_type=None)
        sae_acts_without = sae.encode(cache[hook_name][0, -1, :].unsqueeze(0))[0]

    print(f"\nCase {i}: {int(age)}yo {sex_word}")
    print(f"  {'Feature':<10} {'With Demo':<15} {'Without Demo':<15} {'Diff':<15} {'Claimed for':<15}")
    print(f"  {'-'*70}")

    for feat_idx in sorted(all_features):
        val_with = sae_acts_with[feat_idx].item()
        val_without = sae_acts_without[feat_idx].item()
        diff = val_with - val_without

        groups = []
        if feat_idx in MALE_FEATURES_WITH_DIRECTIONS: groups.append("male")
        if feat_idx in FEMALE_FEATURES_WITH_DIRECTIONS: groups.append("female")
        if feat_idx in YOUNG_FEATURES_WITH_DIRECTIONS: groups.append("young")
        if feat_idx in OLD_FEATURES_WITH_DIRECTIONS: groups.append("old")

        if abs(val_with) > 0.01 or abs(val_without) > 0.01 or abs(diff) > 0.01:
            print(f"  {feat_idx:<10} {val_with:<15.6f} {val_without:<15.6f} {diff:<+15.6f} {','.join(groups)}")

    n_active = sum(1 for f in all_features if abs(sae_acts_with[f].item()) > 0.01)
    print(f"  Active features (>0.01): {n_active}/{len(all_features)}")


Case 0: 8yo male
  Feature    With Demo       Without Demo    Diff            Claimed for    
  ----------------------------------------------------------------------
  Active features (>0.01): 0/32

Case 1: 31yo female
  Feature    With Demo       Without Demo    Diff            Claimed for    
  ----------------------------------------------------------------------
  Active features (>0.01): 0/32

Case 2: 25yo female
  Feature    With Demo       Without Demo    Diff            Claimed for    
  ----------------------------------------------------------------------
  Active features (>0.01): 0/32


## 6. Run the validation test

This is the core experiment. For each case, we compare four conditions.

In [23]:
results = []
test_cases = [c for c in cases if c.get("age") and c.get("sex")][:NUM_TEST_CASES]
print(f"Running validation on {len(test_cases)} cases...")
print("=" * 100)

for i, case in enumerate(test_cases):
    age = case["age"]
    sex = case["sex"]
    sex_word = "male" if sex == "M" else "female"
    diagnosis = case.get("diagnosis", "Unknown")
    symptoms_text = symptoms_to_text(case["features"], evidences)

    prompt_with_demo = f"A {sex_word} {age}-year-old patient has symptoms: {symptoms_text}."
    prompt_without_demo = f"A patient has symptoms: {symptoms_text}."

    # Build clamp targets
    if sex == "M":
        sex_features = {idx: vals["male_mean"] for idx, vals in MALE_FEATURES_WITH_DIRECTIONS.items()}
    else:
        sex_features = {idx: vals["female_mean"] for idx, vals in FEMALE_FEATURES_WITH_DIRECTIONS.items()}

    age_val = int(age)
    if age_val < 18:
        age_features = {idx: vals["young_mean"] for idx, vals in YOUNG_FEATURES_WITH_DIRECTIONS.items()}
    elif age_val >= 60:
        age_features = {idx: vals["old_mean"] for idx, vals in OLD_FEATURES_WITH_DIRECTIONS.items()}
    else:
        age_features = {}

    all_clamp_features = {**sex_features, **age_features}

    print(f"\nCase {i}: {age}yo {sex_word}, True Dx: {diagnosis}")
    print(f"  Clamping {len(sex_features)} sex + {len(age_features)} age features")

    # [A] With demographics in plain text
    t0 = time.time()
    scores_a = score_all_diagnoses(prompt_with_demo, model, diagnosis_list)
    top5_a = [dx for dx, _ in scores_a[:5]]
    print(f"  [A] With demo:          {top5_a[0]:<40} ({time.time()-t0:.1f}s)")

    # [B] No demo + set-value clamping
    t0 = time.time()
    scores_b = score_all_diagnoses(
        prompt_without_demo, model, diagnosis_list,
        sae=sae, clamp_features=all_clamp_features, clamp_method="set_value",
    )
    top5_b = [dx for dx, _ in scores_b[:5]]
    print(f"  [B] Set-value clamp:    {top5_b[0]:<40} ({time.time()-t0:.1f}s)")

    # [C] No demo + multiplicative 5x
    t0 = time.time()
    scores_c = score_all_diagnoses(
        prompt_without_demo, model, diagnosis_list,
        sae=sae, clamp_features=all_clamp_features,
        clamp_method="multiplicative", clamping_level=5.0,
    )
    top5_c = [dx for dx, _ in scores_c[:5]]
    print(f"  [C] Mult 5x clamp:      {top5_c[0]:<40} ({time.time()-t0:.1f}s)")

    # [D] No demo, no clamping
    t0 = time.time()
    scores_d = score_all_diagnoses(prompt_without_demo, model, diagnosis_list)
    top5_d = [dx for dx, _ in scores_d[:5]]
    print(f"  [D] No clamp (neutral): {top5_d[0]:<40} ({time.time()-t0:.1f}s)")

    j_b = jaccard(top5_a, top5_b)
    j_c = jaccard(top5_a, top5_c)
    j_d = jaccard(top5_a, top5_d)
    print(f"  Jaccard vs [A]:  [B]={j_b:.2f}  [C]={j_c:.2f}  [D]={j_d:.2f}")
    print(f"  Top-1 match:     [B]={top5_a[0]==top5_b[0]}  [C]={top5_a[0]==top5_c[0]}  [D]={top5_a[0]==top5_d[0]}")

    results.append({
        "case_id": i, "age": age, "sex": sex_word, "true_dx": diagnosis,
        "top1_with_demo": top5_a[0], "top1_setval": top5_b[0],
        "top1_mult": top5_c[0], "top1_neutral": top5_d[0],
        "jaccard_setval": j_b, "jaccard_mult": j_c, "jaccard_neutral": j_d,
        "top1_match_setval": top5_a[0] == top5_b[0],
        "top1_match_mult": top5_a[0] == top5_c[0],
        "top1_match_neutral": top5_a[0] == top5_d[0],
    })

Running validation on 5 cases...

Case 0: 8yo male, True Dx: Pneumonia
  Clamping 15 sex + 5 age features
  [A] With demo:          Guillain-Barré syndrome                  (12.9s)
  [B] Set-value clamp:    Guillain-Barré syndrome                  (31.9s)
  [C] Mult 5x clamp:      Guillain-Barré syndrome                  (37.0s)
  [D] No clamp (neutral): Guillain-Barré syndrome                  (12.8s)
  Jaccard vs [A]:  [B]=0.43  [C]=0.43  [D]=1.00
  Top-1 match:     [B]=True  [C]=True  [D]=True

Case 1: 31yo female, True Dx: Anaphylaxis
  Clamping 15 sex + 0 age features
  [A] With demo:          Guillain-Barré syndrome                  (12.2s)
  [B] Set-value clamp:    Bronchospasm / acute asthma exacerbation (26.6s)
  [C] Mult 5x clamp:      Bronchospasm / acute asthma exacerbation (30.1s)
  [D] No clamp (neutral): Guillain-Barré syndrome                  (12.1s)
  Jaccard vs [A]:  [B]=0.67  [C]=0.67  [D]=1.00
  Top-1 match:     [B]=False  [C]=False  [D]=True

Case 2: 25yo female, 

## 7. Summary

In [24]:
n = len(results)
print("=" * 100)
print("SUMMARY")
print("=" * 100)

setval_match = sum(1 for r in results if r["top1_match_setval"]) / n
mult_match = sum(1 for r in results if r["top1_match_mult"]) / n
neutral_match = sum(1 for r in results if r["top1_match_neutral"]) / n
print(f"\nTop-1 match rate (vs plain text with demo):")
print(f"  Set-value clamping:     {setval_match:.1%}")
print(f"  Multiplicative 5x:      {mult_match:.1%}")
print(f"  No clamping (neutral):  {neutral_match:.1%}")

mean_j_b = np.mean([r["jaccard_setval"] for r in results])
mean_j_c = np.mean([r["jaccard_mult"] for r in results])
mean_j_d = np.mean([r["jaccard_neutral"] for r in results])
print(f"\nMean Top-5 Jaccard (vs plain text with demo):")
print(f"  Set-value clamping:     {mean_j_b:.3f}")
print(f"  Multiplicative 5x:      {mean_j_c:.3f}")
print(f"  No clamping (neutral):  {mean_j_d:.3f}")

setval_eq_neutral = sum(1 for r in results if r["top1_setval"] == r["top1_neutral"]) / n
mult_eq_neutral = sum(1 for r in results if r["top1_mult"] == r["top1_neutral"]) / n
print(f"\nCases where clamped = neutral (features had zero effect):")
print(f"  Set-value: {setval_eq_neutral:.1%}")
print(f"  Multiplicative: {mult_eq_neutral:.1%}")

print(f"\nINTERPRETATION:")
if mean_j_b > mean_j_d + 0.05:
    print(f"  Set-value clamping shifts diagnoses toward the demographic.")
    print(f"  Features appear to encode demographic information.")
elif mean_j_b > mean_j_d - 0.02:
    print(f"  Set-value clamping has minimal effect vs neutral baseline.")
    print(f"  Features may not encode demographics, or encode-modify-decode is lossy.")
else:
    print(f"  Set-value clamping is WORSE than neutral.")
    print(f"  Features are likely incorrect.")

SUMMARY

Top-1 match rate (vs plain text with demo):
  Set-value clamping:     40.0%
  Multiplicative 5x:      40.0%
  No clamping (neutral):  100.0%

Mean Top-5 Jaccard (vs plain text with demo):
  Set-value clamping:     0.638
  Multiplicative 5x:      0.638
  No clamping (neutral):  1.000

Cases where clamped = neutral (features had zero effect):
  Set-value: 40.0%
  Multiplicative: 40.0%

INTERPRETATION:
  Set-value clamping is WORSE than neutral.
  Features are likely incorrect.


In [25]:
# Save results
df_results = pd.DataFrame(results)
df_results.to_csv("feature_validation_results.csv", index=False)
print("Results saved to feature_validation_results.csv")
df_results

Results saved to feature_validation_results.csv


,case_id,age,sex,true_dx,top1_with_demo,top1_setval,top1_mult,top1_neutral,jaccard_setval,jaccard_mult,jaccard_neutral,top1_match_setval,top1_match_mult,top1_match_neutral
0,0,8,male,Pneumonia,Guillain-Barré syndrome,Guillain-Barré syndrome,Guillain-Barré syndrome,Guillain-Barré syndrome,0.428571,0.428571,1.0,True,True,True
1,1,31,female,Anaphylaxis,Guillain-Barré syndrome,Bronchospasm / acute asthma exacerbation,Bronchospasm / acute asthma exacerbation,Guillain-Barré syndrome,0.666667,0.666667,1.0,False,False,True
2,2,25,female,Chronic rhinosinusitis,Chronic rhinosinusitis,Guillain-Barré syndrome,Guillain-Barré syndrome,Chronic rhinosinusitis,0.666667,0.666667,1.0,False,False,True
3,3,39,female,Myocarditis,Guillain-Barré syndrome,Bronchospasm / acute asthma exacerbation,Bronchospasm / acute asthma exacerbation,Guillain-Barré syndrome,0.428571,0.428571,1.0,False,False,True
4,4,25,male,Pulmonary embolism,Guillain-Barré syndrome,Guillain-Barré syndrome,Guillain-Barré syndrome,Guillain-Barré syndrome,1.000000,1.000000,1.0,True,True,True
